### **Installations, Imports and Paths**

In [ ]:
# ============================================================
# Cell 1 — Installations
# Baselines notebook: no LoRA, no training
# Supports Qwen3 models
# ============================================================

import importlib.util
import subprocess
import sys
from importlib.metadata import version as pkg_version, PackageNotFoundError

def is_installed(import_name):
    return importlib.util.find_spec(import_name) is not None

required = {
    "unsloth": "unsloth",
    "unsloth_zoo": "unsloth_zoo",
    "bitsandbytes": "bitsandbytes",
    "sacrebleu": "sacrebleu",
    "evaluate": "evaluate",
    "datasets": "datasets",
    "transformers": "transformers",
    "accelerate": "accelerate",
    "packaging": "packaging",
}

missing = [
    pip_name
    for pip_name, import_name in required.items()
    if not is_installed(import_name)
]

print("Missing packages:", missing)

if missing:
    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        *missing,
    ]
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)
else:
    print("All required packages are already installed.")

# Qwen3 requires a recent transformers version.
from packaging.version import parse as parse_version

def installed_version(package_name):
    try:
        return pkg_version(package_name)
    except PackageNotFoundError:
        return None

transformers_version = installed_version("transformers")
print("Current transformers:", transformers_version)

if transformers_version is None or parse_version(transformers_version) < parse_version("4.51.0"):
    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "transformers>=4.51.0",
    ]
    print("Upgrading transformers for Qwen3 support:")
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)
else:
    print("transformers version is OK for Qwen3.")

print("Installation finished.")

Missing packages: ['unsloth', 'unsloth_zoo', 'bitsandbytes', 'sacrebleu', 'evaluate']
Running: /usr/bin/python3 -m pip install -q --no-cache-dir unsloth unsloth_zoo bitsandbytes sacrebleu evaluate
Current transformers: 5.5.0
transformers version is OK for Qwen3.
Installation finished.


In [ ]:
# ============================================================
# Cell 2 — Imports and environment check
# ============================================================

import unsloth

import torch
import random
import numpy as np
import pandas as pd
import json
import re
import time
import gc

from pathlib import Path
from tqdm.auto import tqdm
from datasets import load_dataset, get_dataset_config_names, Dataset
import transformers
import accelerate
import sacrebleu

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("WARNING: CUDA is not available.")
    print("Baseline generation on CPU will be very slow.")

print("datasets:", Dataset)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("sacrebleu:", sacrebleu.__version__)
print("Environment check finished.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8
datasets: <class 'datasets.arrow_dataset.Dataset'>
transformers: 5.5.0
accelerate: 1.13.0
sacrebleu: 2.6.0
Environment check finished.


In [ ]:
# ============================================================
# Cell 3 — Mount Drive and define paths
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/alexandria_qwen35_sft")

# Reuse the same prepared_data folder as the finetuning notebooks.
DATA_DIR = PROJECT_DIR / "prepared_data"

# Keep baseline predictions separate.
BASELINE_DIR = PROJECT_DIR / "baselines"
BASELINE_PRED_DIR = BASELINE_DIR / "predictions"

# Finetuned experiment reports are still here.
FINETUNED_PRED_DIR = PROJECT_DIR / "predictions"

for p in [DATA_DIR, BASELINE_DIR, BASELINE_PRED_DIR, FINETUNED_PRED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR:", DATA_DIR)
print("BASELINE_DIR:", BASELINE_DIR)
print("BASELINE_PRED_DIR:", BASELINE_PRED_DIR)
print("FINETUNED_PRED_DIR:", FINETUNED_PRED_DIR)

Mounted at /content/drive
PROJECT_DIR: /content/drive/MyDrive/alexandria_qwen35_sft
DATA_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/prepared_data
BASELINE_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/baselines
BASELINE_PRED_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/baselines/predictions
FINETUNED_PRED_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/predictions


### **Baseline configuration**

In [ ]:
# ============================================================
# Cell 4 — Baseline configuration
# Same dataset/prompt config as finetuning notebooks
# Keeps all previous baseline options
# Adds Qwen3-4B Instruct chat-template options
# Keeps few-shot options
# ============================================================

SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATASET_NAME = "UBC-NLP/alexandria"

SELECTED_CONFIGS_MODE = "EG_ONLY"
MANUAL_CONFIGS = ["EG"]

MAX_CONTEXT_TURNS = 3
USE_PREVIOUS_ENGLISH_CONTEXT = True
USE_METADATA = True

# ------------------------------------------------------------
# Evaluate one baseline at a time to avoid Colab memory issues.
# ------------------------------------------------------------
# Options:
#   "qwen35_2b_base"
#   "qwen3_4b_base"
#   "qwen3_4b_base_complete2shot"
#   "qwen3_4b_instruct"
#   "qwen3_4b_instruct_complete2shot"
#   "qwen3_4b_instruct_chat_v1"
#   "qwen3_4b_instruct_complete2shot_chat_v1"
#   "gemma4_e2b_it_base"
#
# Recommended new path for instruction + complete 2-shot:
ACTIVE_BASELINE_LABEL = "qwen3_4b_instruct_complete2shot_chat_v1"

BASELINE_MODELS = {
    "qwen35_2b_base": {
        "label": "qwen35_2b_base",
        "model_name": "unsloth/Qwen3.5-2B-Base",
        "template_mode": "qwen_manual",
        "load_in_4bit": False,
        "load_in_16bit": False,
        "max_seq_length": 768,
        "max_new_tokens": 120,
        "repetition_penalty": 1.05,
        "no_repeat_ngram_size": None,
        "use_few_shots": False,
        "n_few_shots": 0,
        "max_few_shot_example_chars": None,
    },

    "qwen3_4b_base": {
        "label": "qwen3_4b_base",
        "model_name": "Qwen/Qwen3-4B-Base",
        "template_mode": "qwen_manual",
        "load_in_4bit": True,
        "load_in_16bit": False,
        "max_seq_length": 768,
        "max_new_tokens": 120,
        "repetition_penalty": 1.05,
        "no_repeat_ngram_size": None,
        "use_few_shots": False,
        "n_few_shots": 0,
        "max_few_shot_example_chars": None,
    },

    "qwen3_4b_base_complete2shot": {
        "label": "qwen3_4b_base_complete2shot",
        "model_name": "Qwen/Qwen3-4B-Base",
        "template_mode": "qwen_manual",
        "load_in_4bit": True,
        "load_in_16bit": False,

        # Complete 2-shot trick from Experiment12:
        # examples are complete, selected to be naturally short.
        "max_seq_length": 2048,
        "use_few_shots": True,
        "n_few_shots": 2,
        "max_few_shot_example_chars": 450,

        "max_new_tokens": 120,
        "repetition_penalty": 1.05,
        "no_repeat_ngram_size": None,
    },

    "qwen3_4b_instruct": {
        "label": "qwen3_4b_instruct",
        "model_name": "Qwen/Qwen3-4B-Instruct-2507",

        # Kept for backward compatibility with previous run.
        # Not recommended for the Instruct model.
        "template_mode": "qwen_manual",

        "load_in_4bit": True,
        "load_in_16bit": False,
        "max_seq_length": 768,
        "max_new_tokens": 120,
        "repetition_penalty": 1.05,
        "no_repeat_ngram_size": None,
        "use_few_shots": False,
        "n_few_shots": 0,
        "max_few_shot_example_chars": None,
    },

    "qwen3_4b_instruct_complete2shot": {
        "label": "qwen3_4b_instruct_complete2shot",
        "model_name": "Qwen/Qwen3-4B-Instruct-2507",

        # Kept for backward compatibility with previous run.
        # Not recommended for the Instruct model.
        "template_mode": "qwen_manual",

        "load_in_4bit": True,
        "load_in_16bit": False,

        # Complete 2-shot trick from Experiment12.
        "max_seq_length": 2048,
        "use_few_shots": True,
        "n_few_shots": 2,
        "max_few_shot_example_chars": 450,

        "max_new_tokens": 120,
        "repetition_penalty": 1.05,
        "no_repeat_ngram_size": None,
    },

    "qwen3_4b_instruct_chat_v1": {
        "label": "qwen3_4b_instruct_chat_v1",
        "model_name": "Qwen/Qwen3-4B-Instruct-2507",

        # Correct mode for instruction model.
        "template_mode": "qwen_chat",

        "load_in_4bit": True,
        "load_in_16bit": False,
        "max_seq_length": 768,
        "max_new_tokens": 120,
        "repetition_penalty": 1.05,
        "no_repeat_ngram_size": None,
        "use_few_shots": False,
        "n_few_shots": 0,
        "max_few_shot_example_chars": None,
    },

    "qwen3_4b_instruct_complete2shot_chat_v1": {
        "label": "qwen3_4b_instruct_complete2shot_chat_v1",
        "model_name": "Qwen/Qwen3-4B-Instruct-2507",

        # Correct mode for instruction model.
        # This creates a new prediction path and avoids overwriting old manual-format outputs.
        "template_mode": "qwen_chat",

        "load_in_4bit": True,
        "load_in_16bit": False,

        # Same complete 2-shot trick from Experiment12:
        # examples are complete, selected to be naturally short.
        "max_seq_length": 2048,
        "use_few_shots": True,
        "n_few_shots": 2,
        "max_few_shot_example_chars": 450,

        # Keep 120 first. Increase only if clean outputs are genuinely truncated.
        "max_new_tokens": 120,
        "repetition_penalty": 1.05,
        "no_repeat_ngram_size": None,
    },

    "gemma4_e2b_it_base": {
        "label": "gemma4_e2b_it_base",
        "model_name": "unsloth/gemma-4-E2B-it",
        "template_mode": "gemma_native",
        "load_in_4bit": True,
        "load_in_16bit": False,
        "max_seq_length": 768,
        "max_new_tokens": 96,
        "repetition_penalty": 1.10,
        "no_repeat_ngram_size": 4,
        "use_few_shots": False,
        "n_few_shots": 0,
        "max_few_shot_example_chars": None,
    },
}

if ACTIVE_BASELINE_LABEL not in BASELINE_MODELS:
    raise ValueError(
        f"Unknown ACTIVE_BASELINE_LABEL={ACTIVE_BASELINE_LABEL}. "
        f"Available: {list(BASELINE_MODELS.keys())}"
    )

BASELINE_CFG = BASELINE_MODELS[ACTIVE_BASELINE_LABEL]

MODEL_NAME = BASELINE_CFG["model_name"]

MAX_SEQ_LENGTH = BASELINE_CFG["max_seq_length"]

USE_FEW_SHOTS = BASELINE_CFG.get("use_few_shots", False)
N_FEW_SHOTS = BASELINE_CFG.get("n_few_shots", 0)
MAX_FEW_SHOT_EXAMPLE_CHARS = BASELINE_CFG.get("max_few_shot_example_chars", None)

EXPERIMENT_NAME = (
    f"baseline_no_finetune_{ACTIVE_BASELINE_LABEL}_"
    f"alexandria_{SELECTED_CONFIGS_MODE.lower()}_context{MAX_CONTEXT_TURNS}"
)

print("Active baseline:", ACTIVE_BASELINE_LABEL)
print("Model:", MODEL_NAME)
print("Experiment:", EXPERIMENT_NAME)
print("Template mode:", BASELINE_CFG["template_mode"])
print("Load in 4-bit:", BASELINE_CFG["load_in_4bit"])
print("Max seq length:", MAX_SEQ_LENGTH)
print("Max new tokens:", BASELINE_CFG["max_new_tokens"])
print("Use few-shots:", USE_FEW_SHOTS)
print("N few-shots:", N_FEW_SHOTS)
print("Max few-shot example chars:", MAX_FEW_SHOT_EXAMPLE_CHARS)

expected_pred_path = (
    BASELINE_PRED_DIR
    / f"full_eval_predictions_{EXPERIMENT_NAME}.csv"
)

expected_tmp_pred_path = (
    BASELINE_PRED_DIR
    / f"full_eval_predictions_{EXPERIMENT_NAME}.partial.csv"
)

expected_metrics_path = (
    BASELINE_PRED_DIR
    / f"full_eval_metrics_{EXPERIMENT_NAME}.json"
)

print("\nExpected prediction path:")
print(expected_pred_path)

print("\nExpected partial prediction path:")
print(expected_tmp_pred_path)

print("\nExpected metrics path:")
print(expected_metrics_path)

Active baseline: qwen3_4b_instruct_complete2shot_chat_v1
Model: Qwen/Qwen3-4B-Instruct-2507
Experiment: baseline_no_finetune_qwen3_4b_instruct_complete2shot_chat_v1_alexandria_eg_only_context3
Template mode: qwen_chat
Load in 4-bit: True
Max seq length: 2048
Max new tokens: 120
Use few-shots: True
N few-shots: 2
Max few-shot example chars: 450

Expected prediction path:
/content/drive/MyDrive/alexandria_qwen35_sft/baselines/predictions/full_eval_predictions_baseline_no_finetune_qwen3_4b_instruct_complete2shot_chat_v1_alexandria_eg_only_context3.csv

Expected partial prediction path:
/content/drive/MyDrive/alexandria_qwen35_sft/baselines/predictions/full_eval_predictions_baseline_no_finetune_qwen3_4b_instruct_complete2shot_chat_v1_alexandria_eg_only_context3.partial.csv

Expected metrics path:
/content/drive/MyDrive/alexandria_qwen35_sft/baselines/predictions/full_eval_metrics_baseline_no_finetune_qwen3_4b_instruct_complete2shot_chat_v1_alexandria_eg_only_context3.json


### **Data Download and preparation**

In [ ]:
# ============================================================
# Cell 5 — List Alexandria configs and load selected configs
# Exact same config logic as finetuning notebooks
# ============================================================

available_configs = get_dataset_config_names(DATASET_NAME)

print("Available Alexandria configs:")
print(available_configs)

if SELECTED_CONFIGS_MODE == "EG_ONLY":
    selected_configs = ["EG"] if "EG" in available_configs else [available_configs[0]]

elif SELECTED_CONFIGS_MODE == "ALL":
    selected_configs = available_configs

elif SELECTED_CONFIGS_MODE == "MANUAL":
    selected_configs = MANUAL_CONFIGS
    missing = [c for c in selected_configs if c not in available_configs]
    if missing:
        raise ValueError(f"These configs are not available: {missing}")

else:
    raise ValueError("SELECTED_CONFIGS_MODE must be EG_ONLY, ALL, or MANUAL.")

print("\nSelected configs:")
print(selected_configs)

loaded = {}

for cfg in selected_configs:
    print(f"\nLoading config: {cfg}")
    ds_train = load_dataset(DATASET_NAME, name=cfg, split="train")
    ds_test  = load_dataset(DATASET_NAME, name=cfg, split="test")

    loaded[cfg] = {
        "train": ds_train,
        "test": ds_test,
    }

    print("Train:", ds_train)
    print("Test:", ds_test)
    print("Example keys:", ds_train[0].keys())

README.md:   0%|          | 0.00/24.5k [00:00<?, ?B/s]

Available Alexandria configs:
['EG', 'JO', 'LB', 'LY', 'MA', 'MR', 'OM', 'PS', 'SA', 'SD', 'SY', 'TN', 'YE']

Selected configs:
['EG']

Loading config: EG


EG/train-00000-of-00001.parquet:   0%|          | 0.00/496k [00:00<?, ?B/s]

EG/test-00000-of-00001.parquet:   0%|          | 0.00/196k [00:00<?, ?B/s]

EG/dev-00000-of-00001.parquet:   0%|          | 0.00/183k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/982 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/366 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/352 [00:00<?, ? examples/s]

Train: Dataset({
    features: ['conv_id', 'country', 'domain', 'dialect', 'participants', 'english_conversation', 'dialectal_conversation', 'translator_id', 'reviewer_id'],
    num_rows: 982
})
Test: Dataset({
    features: ['conv_id', 'country', 'domain', 'dialect', 'participants', 'english_conversation', 'dialectal_conversation', 'translator_id', 'reviewer_id'],
    num_rows: 366
})
Example keys: dict_keys(['conv_id', 'country', 'domain', 'dialect', 'participants', 'english_conversation', 'dialectal_conversation', 'translator_id', 'reviewer_id'])


In [ ]:
# ============================================================
# Cell 6 — Helper functions for robust extraction
# Exact same as finetuning notebooks
# ============================================================

def safe_get(row, keys, default=""):
    for k in keys:
        if isinstance(row, dict) and k in row and row[k] is not None:
            return row[k]
    return default

def turn_text(turn):
    if isinstance(turn, dict):
        for k in ["text", "sentence", "utterance", "content", "value"]:
            if k in turn and turn[k] is not None:
                return str(turn[k]).strip()
        return str(turn).strip()
    return str(turn).strip()

def turn_field(turn, keys, default=""):
    if isinstance(turn, dict):
        for k in keys:
            if k in turn and turn[k] is not None:
                return str(turn[k]).strip()
    return default

def normalize_list(x):
    if x is None:
        return []
    if isinstance(x, list):
        return x
    return list(x)

def truncate_text(text, max_chars=1200):
    text = str(text)
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rstrip() + " ..."

In [ ]:
# ============================================================
# Cell 7 — Flatten Alexandria conversations
# Exact same logic as finetuning notebooks
# ============================================================

def flatten_alexandria_split(ds, cfg_name, split_name, max_context_turns=3):
    records = []

    for conv_idx, row in enumerate(ds):
        english_conv = normalize_list(row["english_conversation"])
        dialect_conv = normalize_list(row["dialectal_conversation"])

        n = min(len(english_conv), len(dialect_conv))

        country = safe_get(row, ["country", "country_code"], cfg_name)
        dialect = safe_get(row, ["dialect", "dialect_label", "subdialect", "city", "variety"], "")
        domain = safe_get(row, ["domain", "topic"], "")
        persona = safe_get(row, ["persona", "roles", "speaker_roles"], "")
        conv_id = safe_get(row, ["conversation_id", "id", "dialogue_id"], f"{cfg_name}_{split_name}_{conv_idx}")

        for i in range(n):
            en_turn = english_conv[i]
            ar_turn = dialect_conv[i]

            source_text = turn_text(en_turn)
            target_text = turn_text(ar_turn)

            if not source_text or not target_text:
                continue

            prev_start = max(0, i - max_context_turns)
            prev_en_turns = english_conv[prev_start:i]

            previous_context = []
            for j, t in enumerate(prev_en_turns, start=prev_start):
                previous_context.append({
                    "turn_id": j,
                    "speaker": turn_field(t, ["speaker", "role", "speaker_role"], ""),
                    "direction": turn_field(t, ["direction", "gender_direction", "speaker_addressee_gender"], ""),
                    "text": turn_text(t),
                })

            records.append({
                "source_id": f"{cfg_name}_{split_name}_{conv_id}_{i}",
                "config": cfg_name,
                "split": split_name,
                "conversation_id": conv_id,
                "turn_id": i,

                "country": country,
                "dialect": dialect,
                "domain": domain,
                "persona": persona,

                "speaker": turn_field(en_turn, ["speaker", "role", "speaker_role"], ""),
                "gender_direction": turn_field(en_turn, ["direction", "gender_direction", "speaker_addressee_gender"], ""),

                "previous_english_turns": previous_context,
                "source_text": source_text,
                "target_arabic": target_text,
            })

    return records

train_records = []
eval_records = []

for cfg in selected_configs:
    train_records.extend(
        flatten_alexandria_split(
            loaded[cfg]["train"],
            cfg_name=cfg,
            split_name="train",
            max_context_turns=MAX_CONTEXT_TURNS,
        )
    )

    eval_records.extend(
        flatten_alexandria_split(
            loaded[cfg]["test"],
            cfg_name=cfg,
            split_name="test",
            max_context_turns=MAX_CONTEXT_TURNS,
        )
    )

train_df = pd.DataFrame(train_records)
eval_df = pd.DataFrame(eval_records)

print("Train shape:", train_df.shape)
print("Eval shape:", eval_df.shape)

print("\nTrain configs:")
print(train_df["config"].value_counts())

print("\nEval configs:")
print(eval_df["config"].value_counts())

display(train_df.head())

Train shape: (3108, 14)
Eval shape: (1118, 14)

Train configs:
config
EG    3108
Name: count, dtype: int64

Eval configs:
config
EG    1118
Name: count, dtype: int64


,source_id,config,split,conversation_id,turn_id,country,dialect,domain,persona,speaker,gender_direction,previous_english_turns,source_text,target_arabic
0,EG_train_EG_train_0_0,EG,train,EG_train_0,0,EG,Egyptian Arabic (Cairene) Dialect,Agriculture and farming,,Wholesale Buyer,male -> female,[],Good morning. I'm looking to source 10 tons of...,صباح الخير، عايز عشرة طن من الخرشوف الكويس للت...
1,EG_train_EG_train_0_1,EG,train,EG_train_0,1,EG,Egyptian Arabic (Cairene) Dialect,Agriculture and farming,,Wholesale Seller,female -> male,"[{'turn_id': 0, 'speaker': 'Wholesale Buyer', ...",Good morning to you. You heard correctly. My a...,صباح النور،سمعك مظبوط،الخرشوف بتاعي من أحسن ال...
2,EG_train_EG_train_0_2,EG,train,EG_train_0,2,EG,Egyptian Arabic (Cairene) Dialect,Agriculture and farming,,Wholesale Buyer,male -> female,"[{'turn_id': 0, 'speaker': 'Wholesale Buyer', ...","Excellent. Yes, please show me. I need them to...",ممتاز، لو سمحتي وريني، عايزه بمقاس واحد ومافيه...
3,EG_train_EG_train_0_3,EG,train,EG_train_0,3,EG,Egyptian Arabic (Cairene) Dialect,Agriculture and farming,,Wholesale Seller,female -> male,"[{'turn_id': 0, 'speaker': 'Wholesale Buyer', ...",Don't you worry. You will be very satisfied. M...,متخافش، هتنبسط جدا، سمعتي جاية من الحاجة الكويسة.
4,EG_train_EG_train_1_0,EG,train,EG_train_1,0,EG,Egyptian Arabic (Cairene) Dialect,Agriculture and farming,,Farmer,female -> female,[],I usually use the regular granular fertilizer....,أنا عادة بستخدم السماد العادي الحبيبات. ايه فا...


In [ ]:
# ============================================================
# Cell 8 — Save prepared flattened data
# Same filenames as finetuning notebooks
# ============================================================

train_jsonl = DATA_DIR / f"alexandria_train_{SELECTED_CONFIGS_MODE.lower()}_context{MAX_CONTEXT_TURNS}.jsonl"
eval_jsonl  = DATA_DIR / f"alexandria_eval_{SELECTED_CONFIGS_MODE.lower()}_context{MAX_CONTEXT_TURNS}.jsonl"

train_df.to_json(train_jsonl, orient="records", lines=True, force_ascii=False)
eval_df.to_json(eval_jsonl, orient="records", lines=True, force_ascii=False)

print("Saved train:", train_jsonl)
print("Saved eval:", eval_jsonl)

Saved train: /content/drive/MyDrive/alexandria_qwen35_sft/prepared_data/alexandria_train_eg_only_context3.jsonl
Saved eval: /content/drive/MyDrive/alexandria_qwen35_sft/prepared_data/alexandria_eval_eg_only_context3.jsonl


### **Building prompts**

In [ ]:
# ============================================================
# Cell 9 — Build prompt and chat messages
# Same old baseline prompt for normal baselines
# Same Experiment12 complete-2shot prompt/selection for few-shot baselines
# Works for Qwen3-4B Base and Qwen3-4B Instruct options
# ============================================================

from datasets import Dataset
import hashlib
import pandas as pd

SYSTEM_PROMPT_NO_FEWSHOT = (
    "You are a professional machine translation system. "
    "Translate the current English dialogue turn into natural dialectal Arabic. "
    "Return only the translation, without explanation."
)

# Same as Experiment12.
SYSTEM_PROMPT_FEWSHOT = (
    "You are a professional machine translation system. "
    "Translate the current English dialogue turn into natural dialectal Arabic. "
    "Use the provided training examples only as style and dialect guidance. "
    "Return only the translation, without explanation."
)

SYSTEM_PROMPT = SYSTEM_PROMPT_FEWSHOT if USE_FEW_SHOTS else SYSTEM_PROMPT_NO_FEWSHOT

def build_context(previous_turns):
    if not USE_PREVIOUS_ENGLISH_CONTEXT or not previous_turns:
        return "No previous context."

    lines = []
    for i, t in enumerate(previous_turns, start=1):
        speaker = t.get("speaker", "")
        text = t.get("text", "")

        if speaker:
            lines.append(f"{i}. {speaker}: {text}")
        else:
            lines.append(f"{i}. {text}")

    return "\n".join(lines)

def build_metadata_block(row):
    if not USE_METADATA:
        return "No metadata."

    fields = [
        ("Country/config", row.get("config", "")),
        ("Target dialect", row.get("dialect", "")),
        ("Domain", row.get("domain", "")),
        ("Persona/Roles", row.get("persona", "")),
        ("Current speaker", row.get("speaker", "")),
        ("Speaker-to-addressee gender direction", row.get("gender_direction", "")),
    ]

    lines = []
    for k, v in fields:
        v = str(v).strip()
        if v:
            lines.append(f"{k}: {v}")

    return "\n".join(lines) if lines else "No metadata."

# ------------------------------------------------------------
# Experiment12 exact complete-2shot selection helpers
# ------------------------------------------------------------

def deterministic_seed_from_id(source_id, base_seed=SEED):
    raw = f"{source_id}_{base_seed}".encode("utf-8")
    return int(hashlib.md5(raw).hexdigest()[:8], 16)

def select_two_shots_from_train(row, train_pool, n=N_FEW_SHOTS):
    """
    Select n COMPLETE examples from training data only.

    The examples are NOT truncated.

    To keep MAX_SEQ_LENGTH reasonable, we prefer naturally short examples:
        source_text length + target_arabic length <= MAX_FEW_SHOT_EXAMPLE_CHARS

    Priority:
    1. same config + same domain + short
    2. same config + short
    3. any short
    4. same config + same domain
    5. same config
    6. any training example

    For train rows, exclude the same source_id to avoid using itself as a shot.
    """
    if not USE_FEW_SHOTS or n <= 0:
        return []

    row_source_id = str(row.get("source_id", ""))
    row_config = str(row.get("config", ""))
    row_domain = str(row.get("domain", ""))

    pool = train_pool.copy()
    pool["source_id"] = pool["source_id"].astype(str)

    # Avoid using the same train row as its own few-shot example.
    pool = pool[pool["source_id"] != row_source_id].copy()

    if len(pool) == 0:
        return []

    pool["fewshot_total_chars"] = (
        pool["source_text"].astype(str).str.len()
        + pool["target_arabic"].astype(str).str.len()
    )

    if MAX_FEW_SHOT_EXAMPLE_CHARS is not None:
        short_pool = pool[
            pool["fewshot_total_chars"] <= MAX_FEW_SHOT_EXAMPLE_CHARS
        ].copy()
    else:
        short_pool = pool.iloc[0:0].copy()

    same_config_domain_short = short_pool[
        (short_pool["config"].astype(str) == row_config)
        & (short_pool["domain"].astype(str) == row_domain)
    ]

    same_config_short = short_pool[
        short_pool["config"].astype(str) == row_config
    ]

    same_config_domain = pool[
        (pool["config"].astype(str) == row_config)
        & (pool["domain"].astype(str) == row_domain)
    ]

    same_config = pool[
        pool["config"].astype(str) == row_config
    ]

    candidate_pools = [
        same_config_domain_short,
        same_config_short,
        short_pool,
        same_config_domain,
        same_config,
        pool,
    ]

    candidates = None
    for candidate_pool in candidate_pools:
        if len(candidate_pool) >= n:
            candidates = candidate_pool
            break

    if candidates is None:
        candidates = pool

    sample_n = min(n, len(candidates))
    seed = deterministic_seed_from_id(row_source_id)

    shots = candidates.sample(n=sample_n, random_state=seed)

    keep_cols = [
        "source_id",
        "config",
        "dialect",
        "domain",
        "source_text",
        "target_arabic",
    ]

    return shots[keep_cols].to_dict("records")

def build_few_shot_block(few_shot_examples):
    if not USE_FEW_SHOTS or not few_shot_examples:
        return "No examples available."

    blocks = []

    for i, ex in enumerate(few_shot_examples, start=1):
        ex_config = str(ex.get("config", "")).strip()
        ex_dialect = str(ex.get("dialect", "")).strip()
        ex_domain = str(ex.get("domain", "")).strip()

        meta_parts = []
        if ex_config:
            meta_parts.append(f"config={ex_config}")
        if ex_dialect:
            meta_parts.append(f"dialect={ex_dialect}")
        if ex_domain:
            meta_parts.append(f"domain={ex_domain}")

        meta_line = ", ".join(meta_parts) if meta_parts else "no metadata"

        # COMPLETE examples.
        # No truncation here.
        ex_source = str(ex.get("source_text", "")).strip()
        ex_target = str(ex.get("target_arabic", "")).strip()

        blocks.append(
            f"""Example {i} ({meta_line})
English:
{ex_source}

Arabic:
{ex_target}"""
        )

    return "\n\n".join(blocks)

# ------------------------------------------------------------
# Prompt builders
# ------------------------------------------------------------

def make_user_prompt_no_fewshot(row):
    context = build_context(row["previous_english_turns"])
    metadata = build_metadata_block(row)

    return f"""Task:
Translate the current English dialogue turn into the target dialectal Arabic variety.

Metadata:
{metadata}

Previous English dialogue context:
{context}

Current English turn:
{row["source_text"]}

Rules:
- Preserve the meaning exactly.
- Use natural local dialectal Arabic, not Modern Standard Arabic unless it is natural in context.
- Preserve names, numbers, named entities, and technical terms when appropriate.
- Keep the tone appropriate for the speaker and domain.
- Return only the Arabic translation."""

def make_user_prompt_fewshot(row):
    # Same as Experiment12.
    context = build_context(row["previous_english_turns"])
    metadata = build_metadata_block(row)
    few_shots = build_few_shot_block(row.get("few_shot_examples", []))

    return f"""Task:
Translate the current English dialogue turn into the target dialectal Arabic variety.

Few-shot training examples:
{few_shots}

Metadata:
{metadata}

Previous English dialogue context:
{context}

Current English turn:
{row["source_text"]}

Rules:
- Preserve the meaning exactly.
- Use the target local dialect, not Modern Standard Arabic unless it is natural in context.
- Follow the dialect/style pattern shown in the few-shot examples when relevant.
- Do not copy the few-shot examples.
- Preserve names, numbers, named entities, and technical terms when appropriate.
- Keep the tone appropriate for the speaker and domain.
- Return only the Arabic translation."""

def make_user_prompt(row):
    if USE_FEW_SHOTS:
        return make_user_prompt_fewshot(row)

    return make_user_prompt_no_fewshot(row)

def row_to_messages(row):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": make_user_prompt(row)},
        {"role": "assistant", "content": row["target_arabic"]},
    ]

# ------------------------------------------------------------
# Build few-shot examples from training data only when enabled
# ------------------------------------------------------------

if USE_FEW_SHOTS:
    few_shot_pool_cols = [
        "source_id",
        "config",
        "dialect",
        "domain",
        "source_text",
        "target_arabic",
    ]

    missing_few_shot_cols = [
        c for c in few_shot_pool_cols
        if c not in train_df.columns
    ]

    if missing_few_shot_cols:
        raise ValueError(
            f"Missing columns in train_df for few-shot selection: {missing_few_shot_cols}"
        )

    train_few_shot_pool = train_df[few_shot_pool_cols].copy()

    train_df["few_shot_examples"] = train_df.apply(
        lambda row: select_two_shots_from_train(
            row,
            train_few_shot_pool,
            n=N_FEW_SHOTS,
        ),
        axis=1,
    )

    eval_df["few_shot_examples"] = eval_df.apply(
        lambda row: select_two_shots_from_train(
            row,
            train_few_shot_pool,
            n=N_FEW_SHOTS,
        ),
        axis=1,
    )

else:
    train_df["few_shot_examples"] = [[] for _ in range(len(train_df))]
    eval_df["few_shot_examples"] = [[] for _ in range(len(eval_df))]

train_df["messages"] = train_df.apply(row_to_messages, axis=1)
eval_df["messages"]  = eval_df.apply(row_to_messages, axis=1)

train_dataset = Dataset.from_pandas(
    train_df[
        [
            "messages",
            "source_id",
            "config",
            "domain",
            "dialect",
            "source_text",
            "target_arabic",
        ]
    ],
    preserve_index=False,
)

eval_dataset = Dataset.from_pandas(
    eval_df[
        [
            "messages",
            "source_id",
            "config",
            "domain",
            "dialect",
            "source_text",
            "target_arabic",
        ]
    ],
    preserve_index=False,
)

print(train_dataset)
print(eval_dataset)

print("\nACTIVE_BASELINE_LABEL:", ACTIVE_BASELINE_LABEL)
print("MODEL_NAME:", MODEL_NAME)
print("USE_FEW_SHOTS:", USE_FEW_SHOTS)
print("N_FEW_SHOTS:", N_FEW_SHOTS)

if USE_FEW_SHOTS:
    print("\nExample few-shot source IDs for first train row:")
    print([x["source_id"] for x in train_df.iloc[0]["few_shot_examples"]])

    print("\nFew-shot example lengths for first train row:")
    for i, ex in enumerate(train_df.iloc[0]["few_shot_examples"], start=1):
        total_chars = len(str(ex["source_text"])) + len(str(ex["target_arabic"]))
        print(f"Example {i}: source_id={ex['source_id']}, total_chars={total_chars}")

print("\nExample messages:")
train_dataset[0]["messages"]

Dataset({
    features: ['messages', 'source_id', 'config', 'domain', 'dialect', 'source_text', 'target_arabic'],
    num_rows: 3108
})
Dataset({
    features: ['messages', 'source_id', 'config', 'domain', 'dialect', 'source_text', 'target_arabic'],
    num_rows: 1118
})

ACTIVE_BASELINE_LABEL: qwen3_4b_instruct_complete2shot_chat_v1
MODEL_NAME: Qwen/Qwen3-4B-Instruct-2507
USE_FEW_SHOTS: True
N_FEW_SHOTS: 2

Example few-shot source IDs for first train row:
['EG_train_EG_train_46_2', 'EG_train_EG_train_67_0']

Few-shot example lengths for first train row:
Example 1: source_id=EG_train_EG_train_46_2, total_chars=243
Example 2: source_id=EG_train_EG_train_67_0, total_chars=75

Example messages:


[{'role': 'system',
  'content': 'You are a professional machine translation system. Translate the current English dialogue turn into natural dialectal Arabic. Use the provided training examples only as style and dialect guidance. Return only the translation, without explanation.'},
 {'role': 'user',
  'content': "Task:\nTranslate the current English dialogue turn into the target dialectal Arabic variety.\n\nFew-shot training examples:\nExample 1 (config=EG, dialect=Egyptian Arabic (Cairene) Dialect, domain=Agriculture and farming)\nEnglish:\nBy monitoring, we'll know exactly when the pest levels are high enough to justify spraying. This saves you money and protects the environment.\n\nArabic:\nلما نراقب، هنعرف بالضبط امتى مستويات الحشرات عالية كفاية عشان نبرر الرش. ده بيوفر فلوس وبيحمي البيئة.\n\nExample 2 (config=EG, dialect=Egyptian Arabic (Cairene) Dialect, domain=Agriculture and farming)\nEnglish:\nSo, what will this cost me for the whole job?\n\nArabic:\nطيب قد ايه الشغل كله هيكل

Sanity Checks

In [ ]:
# ============================================================
# Cell 10 — Optional faithfulness check
# Confirms current eval_df has the same source_ids as existing finetuned prediction files
# ============================================================

def find_latest_csv(pattern, folder):
    files = sorted(folder.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    return files[0] if files else None

expected_ids = set(eval_df["source_id"].astype(str).tolist())

REFERENCE_FINETUNED_EXPERIMENTS = [
    "qwen35_2b_alexandria_eg_only_context3_all_group_r16_v2_10epochs",
    "gemma4_e2b_it_alexandria_eg_only_context3_all_group_r16_native_template_eos_4bit_lr5e5_resume_safe_v2",
]

for exp_name in REFERENCE_FINETUNED_EXPERIMENTS:
    pred_file = find_latest_csv(f"full_eval_predictions_{exp_name}_best_step*.csv", FINETUNED_PRED_DIR)

    print("\nChecking:", exp_name)

    if pred_file is None:
        print("No finetuned prediction file found yet. Skipping.")
        continue

    ref_df = pd.read_csv(pred_file)

    if "source_id" not in ref_df.columns:
        print("Prediction file has no source_id column:", pred_file)
        continue

    actual_ids = set(ref_df["source_id"].astype(str).tolist())

    print("Prediction file:", pred_file)
    print("Current eval examples:", len(expected_ids))
    print("Prediction source_ids:", len(actual_ids))
    print("Missing in prediction:", len(expected_ids - actual_ids))
    print("Extra in prediction:", len(actual_ids - expected_ids))

    if expected_ids == actual_ids:
        print("PASS: source_id set is exactly identical.")
    else:
        print("WARNING: source_id set differs.")


Checking: qwen35_2b_alexandria_eg_only_context3_all_group_r16_v2_10epochs
Prediction file: /content/drive/MyDrive/alexandria_qwen35_sft/predictions/full_eval_predictions_qwen35_2b_alexandria_eg_only_context3_all_group_r16_v2_10epochs_best_step600.csv
Current eval examples: 1118
Prediction source_ids: 1118
Missing in prediction: 0
Extra in prediction: 0
PASS: source_id set is exactly identical.

Checking: gemma4_e2b_it_alexandria_eg_only_context3_all_group_r16_native_template_eos_4bit_lr5e5_resume_safe_v2
Prediction file: /content/drive/MyDrive/alexandria_qwen35_sft/predictions/full_eval_predictions_gemma4_e2b_it_alexandria_eg_only_context3_all_group_r16_native_template_eos_4bit_lr5e5_resume_safe_v2_best_step1000.csv
Current eval examples: 1118
Prediction source_ids: 1118
Missing in prediction: 0
Extra in prediction: 0
PASS: source_id set is exactly identical.


### **Preparing baseline models**

In [ ]:
# ============================================================
# Cell 11 — Load active baseline model
# No LoRA adapter, no finetuning
# ============================================================

try:
    from unsloth import FastLanguageModel
    UnslothModel = FastLanguageModel
    print("Using unsloth.FastLanguageModel")
except Exception as e:
    print("FastLanguageModel import failed:", repr(e))
    from unsloth import FastModel
    UnslothModel = FastModel
    print("Using unsloth.FastModel fallback")

USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

if BASELINE_CFG["load_in_4bit"]:
    dtype = None
else:
    dtype = torch.bfloat16 if USE_BF16 else torch.float16

def load_baseline_model():
    kwargs = dict(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
    )

    try:
        return UnslothModel.from_pretrained(
            **kwargs,
            dtype=dtype,
            load_in_4bit=BASELINE_CFG["load_in_4bit"],
            load_in_16bit=BASELINE_CFG["load_in_16bit"],
        )
    except TypeError:
        return UnslothModel.from_pretrained(
            **kwargs,
            dtype=dtype,
            load_in_4bit=BASELINE_CFG["load_in_4bit"],
        )

model, tokenizer = load_baseline_model()

if hasattr(tokenizer, "tokenizer"):
    print("Tokenizer object has internal tokenizer. Using tokenizer.tokenizer.")
    tokenizer = tokenizer.tokenizer

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

STOP_TOKEN_IDS = []
if tokenizer.eos_token_id is not None:
    STOP_TOKEN_IDS.append(int(tokenizer.eos_token_id))

for tok in ["<end_of_turn>", "<|im_end|>", "<turn|>"]:
    try:
        tok_id = tokenizer.convert_tokens_to_ids(tok)
        if tok_id is not None and tok_id != tokenizer.unk_token_id and tok_id not in STOP_TOKEN_IDS:
            STOP_TOKEN_IDS.append(int(tok_id))
    except Exception:
        pass

if len(STOP_TOKEN_IDS) == 0:
    raise RuntimeError("Could not identify EOS/stop token id.")

try:
    UnslothModel.for_inference(model)
except Exception as e:
    print("for_inference not available or not needed:", repr(e))

print("Loaded baseline model:", MODEL_NAME)
print("dtype:", dtype)
print("load_in_4bit:", BASELINE_CFG["load_in_4bit"])
print("Tokenizer type:", type(tokenizer))
print("pad token:", tokenizer.pad_token, tokenizer.pad_token_id)
print("eos token:", tokenizer.eos_token, tokenizer.eos_token_id)
print("stop token ids:", STOP_TOKEN_IDS)
print("chat template exists:", getattr(tokenizer, "chat_template", None) is not None)

Using unsloth.FastLanguageModel
==((====))==  Unsloth 2026.5.10: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.48k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/4.04k [00:00<?, ?B/s]

unsloth/qwen3-4b-instruct-2507-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Loaded baseline model: Qwen/Qwen3-4B-Instruct-2507
dtype: None
load_in_4bit: True
Tokenizer type: <class 'transformers.models.qwen2.tokenization_qwen2.Qwen2Tokenizer'>
pad token: <|PAD_TOKEN|> 151669
eos token: <|im_end|> 151645
stop token ids: [151645]
chat template exists: True


In [ ]:
# ============================================================
# Cell 12 — Model-specific inference formatting
# Qwen Base uses manual SFT template
# Qwen Instruct uses tokenizer.apply_chat_template
# Gemma uses native template style
# Includes output cleaning for prompt leakage
# ============================================================

import torch
import re

SYSTEM_MARKER = "### System:"
INSTRUCTION_MARKER = "### Instruction:"
RESPONSE_MARKER = "### Arabic translation:"

# ------------------------------------------------------------
# General helpers
# ------------------------------------------------------------

def get_message_content(messages, role):
    for m in messages:
        if m.get("role", "") == role:
            return str(m.get("content", ""))
    return ""

def is_qwen_instruct_model():
    model_name = str(MODEL_NAME).lower()
    active_label = str(ACTIVE_BASELINE_LABEL).lower()

    return (
        "instruct" in model_name
        or "instruct" in active_label
    )

def should_use_qwen_chat_template():
    template_mode = str(BASELINE_CFG.get("template_mode", "")).lower()

    return (
        template_mode in ["qwen_chat", "qwen_instruct", "chat"]
        or (
            is_qwen_instruct_model()
            and template_mode not in ["qwen_manual"]
        )
    )

# ------------------------------------------------------------
# Qwen manual format for base models
# ------------------------------------------------------------

def format_sft_text(system_text, user_text, assistant_text=None, add_eos=True):
    text = (
        f"{SYSTEM_MARKER}\n"
        f"{system_text.strip()}\n\n"
        f"{INSTRUCTION_MARKER}\n"
        f"{user_text.strip()}\n\n"
        f"{RESPONSE_MARKER}\n"
    )

    if assistant_text is not None:
        text += assistant_text.strip()

        if add_eos and tokenizer.eos_token is not None:
            text += tokenizer.eos_token

    return text

# ------------------------------------------------------------
# Gemma helper
# ------------------------------------------------------------

def messages_to_gemma_chat(messages, include_assistant=False):
    system_text = ""
    user_text = ""
    assistant_text = ""

    for m in messages:
        role = m.get("role", "")
        content = str(m.get("content", ""))

        if role == "system":
            system_text = content.strip()
        elif role == "user":
            user_text = content.strip()
        elif role in ["assistant", "model"]:
            assistant_text = content.strip()

    folded_user = (
        "System instruction:\n"
        f"{system_text}\n\n"
        "User request:\n"
        f"{user_text}"
    ).strip()

    out = [{"role": "user", "content": folded_user}]

    if include_assistant:
        out.append({"role": "assistant", "content": assistant_text})

    return out

def apply_chat_template_robust(messages, tokenize=False, add_generation_prompt=False):
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=tokenize,
            add_generation_prompt=add_generation_prompt,
        )
    except Exception as e1:
        converted = []

        for m in messages:
            mm = dict(m)

            if mm.get("role") == "assistant":
                mm["role"] = "model"

            converted.append(mm)

        try:
            return tokenizer.apply_chat_template(
                converted,
                tokenize=tokenize,
                add_generation_prompt=add_generation_prompt,
            )
        except Exception as e2:
            print("apply_chat_template failed with assistant role:", repr(e1))
            print("apply_chat_template failed with model role:", repr(e2))
            raise

# ------------------------------------------------------------
# Prompt builder
# ------------------------------------------------------------

def build_prompt_for_row(row):
    template_mode = str(BASELINE_CFG["template_mode"]).lower()

    # Qwen Instruct chat-template path.
    if should_use_qwen_chat_template():
        messages = row["messages"]

        system_text = get_message_content(messages, "system")
        user_text = get_message_content(messages, "user")

        qwen_messages = [
            {"role": "system", "content": system_text.strip()},
            {"role": "user", "content": user_text.strip()},
        ]

        prompt = tokenizer.apply_chat_template(
            qwen_messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        return prompt

    # Qwen Base / manual SFT-style path.
    if template_mode == "qwen_manual":
        user_text = make_user_prompt(row)

        prompt = format_sft_text(
            system_text=SYSTEM_PROMPT,
            user_text=user_text,
            assistant_text=None,
            add_eos=False,
        )

        return prompt

    # Gemma native chat-template path.
    if template_mode == "gemma_native":
        train_like_messages = row_to_messages({
            **row,
            "target_arabic": "",
        })

        gemma_messages = messages_to_gemma_chat(
            train_like_messages,
            include_assistant=False,
        )

        prompt = apply_chat_template_robust(
            gemma_messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        return prompt

    raise ValueError(f"Unknown template_mode: {template_mode}")

# ------------------------------------------------------------
# Output cleaning
# ------------------------------------------------------------

def clean_generation_text(text):
    text = str(text)

    # Remove known control tokens.
    special_tokens = [
        tokenizer.eos_token,
        tokenizer.bos_token,
        tokenizer.pad_token,
        "<end_of_turn>",
        "<start_of_turn>",
        "<|endoftext|>",
        "<|im_end|>",
        "<|im_start|>",
        "<|assistant|>",
        "<|user|>",
        "<|system|>",
        "<turn|>",
    ]

    for tok in special_tokens:
        if tok:
            text = text.replace(tok, "")

    text = text.strip()

    # If the model echoes the manual response marker, keep what follows.
    if RESPONSE_MARKER in text:
        text = text.split(RESPONSE_MARKER, 1)[-1].strip()

    # Remove label if the model starts with a label.
    starting_labels = [
        "الترجمة باللهجة المصرية:",
        "الترجمة المصرية:",
        "الترجمة:",
        "الإجابة:",
        "Arabic:",
        "Egyptian Arabic:",
        "Translation:",
    ]

    for label in starting_labels:
        if text.strip().startswith(label):
            text = text.split(label, 1)[-1].strip()

    # Cut leakage if the model starts a new chat/example/prompt section.
    leakage_markers = [
        "Human:",
        "User:",
        "Assistant:",
        "System:",
        "### System:",
        "### Instruction:",
        "### Arabic translation:",
        "Metadata:",
        "Previous English dialogue context:",
        "Current English turn:",
        "Few-shot training examples:",
        "Example 1",
        "Example 2",
        "English:",
        "Arabic:",
        "الترجمة باللهجة المصرية:",
        "الترجمة المصرية:",
        "الترجمة:",
    ]

    cut_positions = []

    for marker in leakage_markers:
        pos = text.find(marker)

        if pos != -1:
            cut_positions.append(pos)

    if cut_positions:
        first_cut = min(cut_positions)
        text = text[:first_cut].strip()

    # If after cleaning it still starts with a prompt marker, empty it.
    for marker in leakage_markers:
        if text.strip().startswith(marker):
            text = ""
            break

    # Normalize whitespace.
    text = re.sub(r"\s+", " ", text).strip()

    return text

# ------------------------------------------------------------
# Generation
# ------------------------------------------------------------

def generate_translation_from_row(row):
    prompt = build_prompt_for_row(row)

    # Diagnostic token length before truncation.
    n_prompt_tokens_no_trunc = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=False,
        add_special_tokens=False,
    )["input_ids"].shape[-1]

    if n_prompt_tokens_no_trunc > MAX_SEQ_LENGTH:
        print(
            f"WARNING: prompt would truncate: "
            f"{n_prompt_tokens_no_trunc} tokens > MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}"
        )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(model.device)

    input_len = inputs["input_ids"].shape[-1]

    eos_arg = STOP_TOKEN_IDS if len(STOP_TOKEN_IDS) > 1 else STOP_TOKEN_IDS[0]

    gen_kwargs = dict(
        **inputs,
        max_new_tokens=BASELINE_CFG["max_new_tokens"],
        do_sample=False,
        num_beams=1,
        repetition_penalty=BASELINE_CFG["repetition_penalty"],
        eos_token_id=eos_arg,
        pad_token_id=tokenizer.pad_token_id,
        use_cache=True,
    )

    if BASELINE_CFG["no_repeat_ngram_size"] is not None:
        gen_kwargs["no_repeat_ngram_size"] = BASELINE_CFG["no_repeat_ngram_size"]

    with torch.no_grad():
        outputs = model.generate(**gen_kwargs)

    # Decode only generated tokens for every model type.
    generated_ids = outputs[0][input_len:]
    raw_answer = tokenizer.decode(generated_ids, skip_special_tokens=False)

    answer = clean_generation_text(raw_answer)

    full_decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)

    return answer, full_decoded

# ------------------------------------------------------------
# Sanity check
# ------------------------------------------------------------

sample = eval_df.sample(1, random_state=SEED).iloc[0].to_dict()

prompt = build_prompt_for_row(sample)

sample_prompt_tokens = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=False,
    add_special_tokens=False,
)["input_ids"].shape[-1]

pred, raw = generate_translation_from_row(sample)

print("Baseline:", ACTIVE_BASELINE_LABEL)
print("Model:", MODEL_NAME)
print("Template mode:", BASELINE_CFG["template_mode"])
print("Detected Qwen Instruct:", is_qwen_instruct_model())
print("Using Qwen chat template:", should_use_qwen_chat_template())
print("MAX_SEQ_LENGTH:", MAX_SEQ_LENGTH)
print("Prompt tokens without truncation:", sample_prompt_tokens)
print("Would truncate:", sample_prompt_tokens > MAX_SEQ_LENGTH)
print("max_new_tokens:", BASELINE_CFG["max_new_tokens"])

print("\nConfig:", sample["config"])
print("Dialect:", sample["dialect"])
print("Domain:", sample["domain"])

print("\nEnglish:")
print(sample["source_text"])

print("\nReference Arabic:")
print(sample["target_arabic"])

print("\nPrediction:")
print(pred)

print("\nRaw generated/full decoded output for debugging:")
print(raw)

Both `max_new_tokens` (=120) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.1

Baseline: qwen3_4b_instruct_complete2shot_chat_v1
Model: Qwen/Qwen3-4B-Instruct-2507
Template mode: qwen_chat
Detected Qwen Instruct: True
Using Qwen chat template: True
MAX_SEQ_LENGTH: 2048
Prompt tokens without truncation: 459
Would truncate: False
max_new_tokens: 120

Config: EG
Dialect: Egyptian Arabic (Cairene) Dialect
Domain: Construction and real estate

English:
Engineer, good morning. Before you run your cables on the third floor, let's coordinate the wall chases.

Reference Arabic:
صباح الخير يا هندسه. قبل ما تمد الكابلات في الدور التالت، خلينا نتفق على مجاري الحيطان.

Prediction:
مهندس، صباح الخير. قبل أن تبدأ بوضع الكابلات على الطابق الثالث، دعنا نتفق على فتحات الجدران.

Raw generated/full decoded output for debugging:
<|im_start|>system
You are a professional machine translation system. Translate the current English dialogue turn into natural dialectal Arabic. Use the provided training examples only as style and dialect guidance. Return only the translation, without explan

### **Generate full eval predictions**

In [ ]:
# ============================================================
# Cell 13 — Generate predictions on FULL eval/test set
# Safe resume
# ============================================================

EVAL_LIMIT = None
SAVE_EVERY = 25
STORE_RAW_OUTPUT = False
FORCE_REGENERATE_PREDICTIONS = False

pred_path = BASELINE_PRED_DIR / f"full_eval_predictions_{EXPERIMENT_NAME}.csv"
tmp_pred_path = BASELINE_PRED_DIR / f"full_eval_predictions_{EXPERIMENT_NAME}.partial.csv"

print("Baseline:", ACTIVE_BASELINE_LABEL)
print("Model:", MODEL_NAME)
print("Experiment:", EXPERIMENT_NAME)
print("Saving predictions to:", pred_path)
print("Temporary partial file:", tmp_pred_path)

full_eval_df = eval_df.reset_index(drop=True).copy()

if EVAL_LIMIT is not None:
    full_eval_df = full_eval_df.iloc[:EVAL_LIMIT].copy()

expected_n = len(full_eval_df)
expected_ids = set(full_eval_df["source_id"].astype(str).tolist())

print("Total eval/test examples to evaluate:", expected_n)

resume_path = None

if FORCE_REGENERATE_PREDICTIONS:
    print("FORCE_REGENERATE_PREDICTIONS=True")
    pred_rows = []
    done_ids = set()

else:
    if tmp_pred_path.exists():
        resume_path = tmp_pred_path
    elif pred_path.exists():
        resume_path = pred_path

    if resume_path is not None:
        print("Found existing prediction file:")
        print(resume_path)

        existing_df = pd.read_csv(resume_path)

        required_cols = {
            "source_id",
            "prediction",
            "model_name",
            "experiment_name",
        }

        missing_cols = required_cols - set(existing_df.columns)

        if missing_cols:
            print("Existing prediction file is incompatible.")
            print("Missing columns:", missing_cols)
            print("Starting prediction from scratch.")
            pred_rows = []
            done_ids = set()
        else:
            existing_df["source_id"] = existing_df["source_id"].astype(str)
            existing_df = existing_df[existing_df["source_id"].isin(expected_ids)].copy()
            existing_df = existing_df.drop_duplicates(subset=["source_id"], keep="first").copy()

            pred_rows = existing_df.to_dict("records")
            done_ids = set(existing_df["source_id"].astype(str).tolist())

            print("Resuming prediction generation.")
            print("Already completed examples:", len(done_ids))
    else:
        print("No existing prediction file found. Starting from scratch.")
        pred_rows = []
        done_ids = set()

if len(done_ids) == expected_n:
    print("Prediction file already contains all expected eval examples.")
    pred_df = pd.DataFrame(pred_rows)
    pred_df.to_csv(pred_path, index=False, encoding="utf-8-sig")

    if tmp_pred_path.exists():
        try:
            tmp_pred_path.unlink()
            print("Removed partial file after confirming full completion.")
        except Exception as e:
            print("Could not remove partial file:", repr(e))

    display(pred_df.head())

else:
    print(f"Remaining examples to generate: {expected_n - len(done_ids)}")

    start_time = time.time()

    for _, row in tqdm(full_eval_df.iterrows(), total=len(full_eval_df)):
        row_dict = row.to_dict()
        source_id = str(row_dict["source_id"])

        if source_id in done_ids:
            continue

        try:
            pred, raw = generate_translation_from_row(row_dict)

            out_row = {
                "source_id": row_dict["source_id"],
                "config": row_dict.get("config", ""),
                "dialect": row_dict.get("dialect", ""),
                "domain": row_dict.get("domain", ""),
                "source_text": row_dict["source_text"],
                "reference_arabic": row_dict["target_arabic"],
                "prediction": pred,
                "model_name": MODEL_NAME,
                "experiment_name": EXPERIMENT_NAME,
                "baseline_label": ACTIVE_BASELINE_LABEL,
                "template_mode": BASELINE_CFG["template_mode"],
            }

            if STORE_RAW_OUTPUT:
                out_row["raw_output"] = raw

        except Exception as e:
            out_row = {
                "source_id": row_dict.get("source_id", ""),
                "config": row_dict.get("config", ""),
                "dialect": row_dict.get("dialect", ""),
                "domain": row_dict.get("domain", ""),
                "source_text": row_dict.get("source_text", ""),
                "reference_arabic": row_dict.get("target_arabic", ""),
                "prediction": "",
                "generation_error": repr(e),
                "model_name": MODEL_NAME,
                "experiment_name": EXPERIMENT_NAME,
                "baseline_label": ACTIVE_BASELINE_LABEL,
                "template_mode": BASELINE_CFG["template_mode"],
            }

        pred_rows.append(out_row)
        done_ids.add(source_id)

        if len(pred_rows) % SAVE_EVERY == 0:
            tmp_df = pd.DataFrame(pred_rows)
            tmp_df = tmp_df.drop_duplicates(subset=["source_id"], keep="first")
            tmp_df.to_csv(tmp_pred_path, index=False, encoding="utf-8-sig")
            print(f"Saved partial predictions: {len(tmp_df)} rows")

    pred_df = pd.DataFrame(pred_rows)
    pred_df = pred_df.drop_duplicates(subset=["source_id"], keep="first")
    pred_df.to_csv(pred_path, index=False, encoding="utf-8-sig")

    if tmp_pred_path.exists():
        try:
            tmp_pred_path.unlink()
            print("Removed partial prediction file after successful final save.")
        except Exception as e:
            print("Could not remove partial file:", repr(e))

    elapsed = time.time() - start_time

    actual_ids = set(pred_df["source_id"].astype(str).tolist())
    missing_ids = expected_ids - actual_ids
    extra_ids = actual_ids - expected_ids

    print("\nDone.")
    print("Saved predictions to:", pred_path)
    print("Total rows saved:", len(pred_df))
    print("Expected eval/test rows:", expected_n)
    print(f"Elapsed time: {elapsed / 60:.2f} minutes")

    if missing_ids:
        raise RuntimeError(f"Prediction file is incomplete. Missing {len(missing_ids)} eval examples.")

    if extra_ids:
        print(f"Warning: prediction file has {len(extra_ids)} extra source_ids not in current eval_df.")

    if len(pred_df) == expected_n:
        print("Full eval/test set was evaluated successfully.")

    display(pred_df.head())

Baseline: qwen3_4b_instruct_complete2shot_chat_v1
Model: Qwen/Qwen3-4B-Instruct-2507
Experiment: baseline_no_finetune_qwen3_4b_instruct_complete2shot_chat_v1_alexandria_eg_only_context3
Saving predictions to: /content/drive/MyDrive/alexandria_qwen35_sft/baselines/predictions/full_eval_predictions_baseline_no_finetune_qwen3_4b_instruct_complete2shot_chat_v1_alexandria_eg_only_context3.csv
Temporary partial file: /content/drive/MyDrive/alexandria_qwen35_sft/baselines/predictions/full_eval_predictions_baseline_no_finetune_qwen3_4b_instruct_complete2shot_chat_v1_alexandria_eg_only_context3.partial.csv
Total eval/test examples to evaluate: 1118
No existing prediction file found. Starting from scratch.
Remaining examples to generate: 1118


  0%|          | 0/1118 [00:00<?, ?it/s]

Both `max_new_tokens` (=120) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=

Saved partial predictions: 25 rows


Both `max_new_tokens` (=120) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

...

Both `max_new_tokens` (=120) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

Saved partial predictions: 1100 rows


Both `max_new_tokens` (=120) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

Removed partial prediction file after successful final save.

Done.
Saved predictions to: /content/drive/MyDrive/alexandria_qwen35_sft/baselines/predictions/full_eval_predictions_baseline_no_finetune_qwen3_4b_instruct_complete2shot_chat_v1_alexandria_eg_only_context3.csv
Total rows saved: 1118
Expected eval/test rows: 1118
Elapsed time: 41.19 minutes
Full eval/test set was evaluated successfully.


,source_id,config,dialect,domain,source_text,reference_arabic,prediction,model_name,experiment_name,baseline_label,template_mode
0,EG_test_EG_test_0_0,EG,Egyptian Arabic (Cairene) Dialect,Commerce and transactions,"I would like one order of kunafa, please.",عايز واحد كنافة لو سمحت.,أنا أبغى كونافا واحد، يابو.,Qwen/Qwen3-4B-Instruct-2507,baseline_no_finetune_qwen3_4b_instruct_complet...,qwen3_4b_instruct_complete2shot_chat_v1,qwen_chat
1,EG_test_EG_test_0_1,EG,Egyptian Arabic (Cairene) Dialect,Commerce and transactions,Certainly. Would you like that with cheese or ...,أكيد. تحبها بالجبنة ولا بالقشطة؟,بالطبع. تحبها مع الجبن أو مع الكريمة؟,Qwen/Qwen3-4B-Instruct-2507,baseline_no_finetune_qwen3_4b_instruct_complet...,qwen3_4b_instruct_complete2shot_chat_v1,qwen_chat
2,EG_test_EG_test_0_2,EG,Egyptian Arabic (Cairene) Dialect,Commerce and transactions,"With cream, please.",بالقشطة، لو سمحت.,بクリーム، شو.,Qwen/Qwen3-4B-Instruct-2507,baseline_no_finetune_qwen3_4b_instruct_complet...,qwen3_4b_instruct_complete2shot_chat_v1,qwen_chat
3,EG_test_EG_test_1_0,EG,Egyptian Arabic (Cairene) Dialect,Commerce and transactions,"Pardon me, I believe the meat is overcooked. I...",لو سمحت، أعتقد اللحمة مستوية زيادة. ناشفة جدا.,أهلا، أعتقد أن اللحمة مُطهَّاة بدرجة مفرطة، صع...,Qwen/Qwen3-4B-Instruct-2507,baseline_no_finetune_qwen3_4b_instruct_complet...,qwen3_4b_instruct_complete2shot_chat_v1,qwen_chat
4,EG_test_EG_test_1_1,EG,Egyptian Arabic (Cairene) Dialect,Commerce and transactions,"I'm very sorry to hear that, sir. Would you li...",آسفه جدا يا فندم. تحب أخلي الشيف يجهزلك واحدة ...,مافيش مشكلة، يا سيدتي، هل تحب أعمل للحاجة على ...,Qwen/Qwen3-4B-Instruct-2507,baseline_no_finetune_qwen3_4b_instruct_complet...,qwen3_4b_instruct_complete2shot_chat_v1,qwen_chat


### Free GPU before re-run

In [ ]:
# ============================================================
# Cell 15 — Free memory before switching ACTIVE_BASELINE_LABEL
# Run this, then go back to Cell 4, switch ACTIVE_BASELINE_LABEL, and rerun Cells 11–14
# ============================================================

try:
    del model
    del tokenizer
except Exception:
    pass

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("Memory cleaned. You can now switch ACTIVE_BASELINE_LABEL and run the next baseline.")

Memory cleaned. You can now switch ACTIVE_BASELINE_LABEL and run the next baseline.


---

### **final Comparisons with Baselines**
##### **Measuring BLEU, ChrF, and similarity scores**

Run after compute all predictions

In [1]:
# ============================================================
# Cell 24 — Install required packages for final metrics
# BLEU, chrF, chrF++, and E5-large semantic similarity
# ============================================================

import importlib.util
import subprocess
import sys

required = {
    "sentence_transformers": "sentence-transformers",
    "sklearn": "scikit-learn",
    "tqdm": "tqdm",
    "sacrebleu": "sacrebleu",
}

missing = []

for import_name, pip_name in required.items():
    if importlib.util.find_spec(import_name) is None:
        missing.append(pip_name)

print("Missing packages:", missing)

if missing:
    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        *missing,
    ]
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)
else:
    print("All required packages already installed.")

Missing packages: ['sacrebleu']
Running: /usr/bin/python3 -m pip install -q --no-cache-dir sacrebleu


In [2]:
! pip install -q --no-cache-dir sacrebleu

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# ============================================================
# Cell 27 — Recompute final metrics for finetuned systems and baselines
# Includes BLEU, spBLEU, chrF, chrF++, and E5-large semantic similarity
# Includes Qwen3-4B complete-2shot and no-shot all-r16 best_step700
# Saves lexical + semantic metrics consistently
# ============================================================

# ------------------------------------------------------------
# Optional package check for spBLEU SentencePiece tokenization
# ------------------------------------------------------------

import importlib.util
import subprocess
import sys

required_packages = {
    "sacrebleu": "sacrebleu[sentencepiece]",
    "sentencepiece": "sentencepiece",
    "sentence_transformers": "sentence-transformers",
}

missing = []

for import_name, pip_name in required_packages.items():
    if importlib.util.find_spec(import_name) is None:
        missing.append(pip_name)

print("Missing packages:", missing)

if missing:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        *missing,
    ])

# ------------------------------------------------------------
# Imports
# ------------------------------------------------------------

from pathlib import Path
import json
import gc
import pandas as pd
import numpy as np
import torch
import sacrebleu
from sentence_transformers import SentenceTransformer
from IPython.display import display

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROJECT_DIR = Path("/content/drive/MyDrive/alexandria_qwen35_sft")

FINETUNED_PRED_DIR = PROJECT_DIR / "predictions"
BASELINE_PRED_DIR  = PROJECT_DIR / "baselines" / "predictions"

SEMANTIC_OUT_DIR = PROJECT_DIR / "semantic_similarity_e5_large"
SEMANTIC_OUT_DIR.mkdir(parents=True, exist_ok=True)

BASELINE_PRED_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("FINETUNED_PRED_DIR:", FINETUNED_PRED_DIR)
print("BASELINE_PRED_DIR:", BASELINE_PRED_DIR)
print("SEMANTIC_OUT_DIR:", SEMANTIC_OUT_DIR)

# ------------------------------------------------------------
# Finetuned systems
# ------------------------------------------------------------

FINETUNED_METRICS_SPECS = [
    {
        "label": "Qwen3.5-2B LoRA all-r16 best_step600",
        "model_type": "finetuned_lora",
        "metrics_path": FINETUNED_PRED_DIR / (
            "full_eval_metrics_"
            "qwen35_2b_alexandria_eg_only_context3_all_group_r16_v2_10epochs"
            "_best_step600.json"
        ),
        "prediction_path": FINETUNED_PRED_DIR / (
            "full_eval_predictions_"
            "qwen35_2b_alexandria_eg_only_context3_all_group_r16_v2_10epochs"
            "_best_step600.csv"
        ),
    },
    {
        "label": "Gemma-4-E2B-it FNN-r8 MLP best_step2500",
        "model_type": "finetuned_lora",
        "metrics_path": FINETUNED_PRED_DIR / (
            "full_eval_metrics_"
            "gemma4_e2b_it_alexandria_eg_only_context3_"
            "fnn_group_r8_prompt_nat_eg_mlp_r8_lr1e5_10epochs_v1"
            "_best_step2500.json"
        ),
        "prediction_path": FINETUNED_PRED_DIR / (
            "full_eval_predictions_"
            "gemma4_e2b_it_alexandria_eg_only_context3_"
            "fnn_group_r8_prompt_nat_eg_mlp_r8_lr1e5_10epochs_v1"
            "_best_step2500.csv"
        ),
    },
    {
        "label": "Qwen3-4B LoRA complete-2shot all-r16 best_step700",
        "model_type": "finetuned_lora",
        "metrics_path": FINETUNED_PRED_DIR / (
            "full_eval_metrics_"
            "qwen3_4b_alexandria_eg_only_context3_complete2shot_all_group_r16_10epochs"
            "_best_step700.json"
        ),
        "prediction_path": FINETUNED_PRED_DIR / (
            "full_eval_predictions_"
            "qwen3_4b_alexandria_eg_only_context3_complete2shot_all_group_r16_10epochs"
            "_best_step700.csv"
        ),
    },
    {
        "label": "Qwen3-4B LoRA no-shot all-r16 best_step700",
        "model_type": "finetuned_lora",
        "metrics_path": FINETUNED_PRED_DIR / (
            "full_eval_metrics_"
            "qwen3_4b_alexandria_eg_only_context3_noshot_all_group_r16_10epochs_v1"
            "_best_step700.json"
        ),
        "prediction_path": FINETUNED_PRED_DIR / (
            "full_eval_predictions_"
            "qwen3_4b_alexandria_eg_only_context3_noshot_all_group_r16_10epochs_v1"
            "_best_step700.csv"
        ),
    },
]

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

E5_MODEL_NAME = "intfloat/multilingual-e5-large"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# E5-large is heavy; keep batch conservative.
BATCH_SIZE = 8

# spBLEU tokenizer used by FLORES/NLLB-style evaluation.
SPBLEU_TOKENIZER = "flores200"

print("Using device:", DEVICE)
print("E5 model:", E5_MODEL_NAME)
print("Batch size:", BATCH_SIZE)
print("spBLEU tokenizer:", SPBLEU_TOKENIZER)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def read_json(path):
    path = Path(path)
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


def derive_prediction_path_from_metrics_path(metrics_path):
    metrics_path = Path(metrics_path)
    pred_name = metrics_path.name.replace("full_eval_metrics_", "full_eval_predictions_")
    pred_name = pred_name.replace(".json", ".csv")
    return metrics_path.with_name(pred_name)


def derive_metrics_path_from_prediction_path(prediction_path):
    prediction_path = Path(prediction_path)
    metrics_name = prediction_path.name.replace("full_eval_predictions_", "full_eval_metrics_")
    metrics_name = metrics_name.replace(".csv", ".json")
    return prediction_path.with_name(metrics_name)


def safe_first(df, col, default=""):
    if col not in df.columns:
        return default

    vals = df[col].dropna().astype(str).unique()

    if len(vals) == 0:
        return default

    if len(vals) == 1:
        return vals[0]

    return list(vals)


def load_prediction_df(pred_path):
    pred_path = Path(pred_path)

    df = pd.read_csv(pred_path)

    required = {"source_id", "reference_arabic", "prediction"}
    missing = required - set(df.columns)

    if missing:
        raise ValueError(
            f"Missing required columns in {pred_path}: {missing}\n"
            f"Available columns: {list(df.columns)}"
        )

    df["source_id"] = df["source_id"].astype(str)
    df["reference_arabic"] = df["reference_arabic"].fillna("").astype(str)
    df["prediction"] = df["prediction"].fillna("").astype(str)

    df = df.drop_duplicates(subset=["source_id"], keep="first").copy()
    df = df.reset_index(drop=True)

    return df


def compute_spbleu(preds, refs):
    """
    spBLEU = BLEU computed after SentencePiece/FLORES-style tokenization.
    This is commonly used for multilingual MT evaluation.
    """
    try:
        score = sacrebleu.corpus_bleu(
            preds,
            [refs],
            tokenize=SPBLEU_TOKENIZER,
        )
        return float(score.score), SPBLEU_TOKENIZER, ""

    except Exception as e1:
        # Fallback for older SacreBLEU versions that may not support flores200.
        try:
            score = sacrebleu.corpus_bleu(
                preds,
                [refs],
                tokenize="spm",
            )
            return float(score.score), "spm", f"flores200 failed; fallback to spm. Original error: {repr(e1)}"

        except Exception as e2:
            return np.nan, None, f"spBLEU failed. flores200 error: {repr(e1)} | spm error: {repr(e2)}"


def compute_mt_metrics_from_df(df):
    preds = df["prediction"].fillna("").astype(str).tolist()
    refs  = df["reference_arabic"].fillna("").astype(str).tolist()

    bleu   = sacrebleu.corpus_bleu(preds, [refs])
    chrf   = sacrebleu.corpus_chrf(preds, [refs], word_order=0)
    chrfpp = sacrebleu.corpus_chrf(preds, [refs], word_order=2)

    spbleu_score, spbleu_tokenizer, spbleu_error = compute_spbleu(preds, refs)

    return {
        "BLEU": float(bleu.score),
        "spBLEU": float(spbleu_score) if not pd.isna(spbleu_score) else np.nan,
        "spBLEU_tokenizer": spbleu_tokenizer,
        "spBLEU_error": spbleu_error,
        "chrF": float(chrf.score),
        "chrF++": float(chrfpp.score),
    }


def e5_format(text):
    return "passage: " + str(text).strip()


def compute_e5_pairwise_similarity(df, model):
    refs = [e5_format(x) for x in df["reference_arabic"].tolist()]
    preds = [e5_format(x) for x in df["prediction"].tolist()]

    ref_emb = model.encode(
        refs,
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True,
    )

    pred_emb = model.encode(
        preds,
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True,
    )

    # Embeddings are normalized, so dot product = cosine similarity.
    sims = np.sum(ref_emb * pred_emb, axis=1)

    return sims


def summarize_similarity(scores):
    scores = np.asarray(scores, dtype=float)

    return {
        "E5_large_cosine_mean": float(np.mean(scores)),
        "E5_large_cosine_median": float(np.median(scores)),
        "E5_large_cosine_std": float(np.std(scores)),
        "E5_large_cosine_min": float(np.min(scores)),
        "E5_large_cosine_max": float(np.max(scores)),
        "E5_large_cosine_p05": float(np.percentile(scores, 5)),
        "E5_large_cosine_p25": float(np.percentile(scores, 25)),
        "E5_large_cosine_p75": float(np.percentile(scores, 75)),
        "E5_large_cosine_p95": float(np.percentile(scores, 95)),
    }


def compute_group_metrics(df, group_col):
    if group_col not in df.columns:
        return []

    rows = []

    for value in sorted(df[group_col].dropna().astype(str).unique()):
        tmp = df[df[group_col].astype(str) == value].copy()

        if len(tmp) == 0:
            continue

        mt_scores = compute_mt_metrics_from_df(tmp)
        semantic_scores = summarize_similarity(tmp["E5_large_cosine"].values)

        rows.append({
            group_col: value,
            "num_examples": int(len(tmp)),
            **mt_scores,
            **semantic_scores,
        })

    return rows


def make_exp_record_from_finetuned_spec(spec):
    label = spec["label"]
    model_type = spec["model_type"]
    metrics_path = Path(spec["metrics_path"])
    prediction_path = Path(spec["prediction_path"])

    metrics_data = read_json(metrics_path)

    # Prefer explicit prediction_path from spec.
    if prediction_path.exists():
        return {
            "label": label,
            "model_type": model_type,
            "metrics_path": str(metrics_path),
            "prediction_file": str(prediction_path),
            "metrics_data": metrics_data or {},
            "found": True,
            "error": "",
        }

    # Fallback: if metrics file exists and contains prediction_file.
    if metrics_data is not None:
        pred_from_json = metrics_data.get("prediction_file", None)

        if pred_from_json is not None and Path(pred_from_json).exists():
            return {
                "label": label,
                "model_type": model_type,
                "metrics_path": str(metrics_path),
                "prediction_file": str(Path(pred_from_json)),
                "metrics_data": metrics_data,
                "found": True,
                "error": "",
            }

    # Fallback: derive prediction from metrics name.
    derived_pred = derive_prediction_path_from_metrics_path(metrics_path)

    if derived_pred.exists():
        return {
            "label": label,
            "model_type": model_type,
            "metrics_path": str(metrics_path),
            "prediction_file": str(derived_pred),
            "metrics_data": metrics_data or {},
            "found": True,
            "error": "",
        }

    return {
        "label": label,
        "model_type": model_type,
        "metrics_path": str(metrics_path),
        "prediction_file": str(prediction_path),
        "metrics_data": metrics_data or {},
        "found": False,
        "error": (
            "Prediction file not found. Checked explicit prediction_path, "
            "prediction_file inside JSON, and derived prediction path."
        ),
    }


def make_exp_record_from_baseline_csv(pred_path):
    pred_path = Path(pred_path)
    label = pred_path.stem.replace("full_eval_predictions_", "")

    return {
        "label": label,
        "model_type": "baseline_no_finetune",
        "metrics_path": None,
        "prediction_file": str(pred_path),
        "metrics_data": {},
        "found": True,
        "error": "",
    }


def infer_baseline_metrics_path(experiment_name):
    return BASELINE_PRED_DIR / f"full_eval_metrics_{experiment_name}.json"


def infer_experiment_name_from_prediction_path(pred_path):
    pred_path = Path(pred_path)
    return pred_path.stem.replace("full_eval_predictions_", "")


# ------------------------------------------------------------
# Collect experiments
# ------------------------------------------------------------

experiments = []

for spec in FINETUNED_METRICS_SPECS:
    experiments.append(make_exp_record_from_finetuned_spec(spec))

baseline_pred_files = sorted(
    p for p in BASELINE_PRED_DIR.glob("full_eval_predictions_baseline_no_finetune_*.csv")
    if ".partial" not in p.name
)

for pred_file in baseline_pred_files:
    experiments.append(make_exp_record_from_baseline_csv(pred_file))

print("\nExperiments to evaluate:")
for exp in experiments:
    print("-", exp["label"], "| found:", exp["found"], "| pred:", exp["prediction_file"])

qwen3_complete2shot_found = any(
    exp["prediction_file"] is not None
    and "qwen3_4b_alexandria_eg_only_context3_complete2shot_all_group_r16_10epochs_best_step700" in str(exp["prediction_file"])
    for exp in experiments
)

print("\nQwen3-4B complete-2shot finetuned prediction found:", qwen3_complete2shot_found)

if not qwen3_complete2shot_found:
    print(
        "WARNING: No Qwen3-4B complete-2shot finetuned prediction CSV was found. Expected:\n"
        f"{FINETUNED_PRED_DIR}/full_eval_predictions_"
        "qwen3_4b_alexandria_eg_only_context3_complete2shot_all_group_r16_10epochs"
        "_best_step700.csv"
    )

qwen3_noshot_found = any(
    exp["prediction_file"] is not None
    and "qwen3_4b_alexandria_eg_only_context3_noshot_all_group_r16_10epochs_v1_best_step700" in str(exp["prediction_file"])
    for exp in experiments
)

print("Qwen3-4B no-shot finetuned prediction found:", qwen3_noshot_found)

if not qwen3_noshot_found:
    print(
        "WARNING: No Qwen3-4B no-shot finetuned prediction CSV was found. Expected:\n"
        f"{FINETUNED_PRED_DIR}/full_eval_predictions_"
        "qwen3_4b_alexandria_eg_only_context3_noshot_all_group_r16_10epochs_v1"
        "_best_step700.csv"
    )

qwen3_baseline_found = any(
    exp["prediction_file"] is not None
    and "qwen3_4b_base" in str(exp["prediction_file"])
    for exp in experiments
)

print("Qwen3-4B baseline prediction found:", qwen3_baseline_found)

# ------------------------------------------------------------
# Load E5-large
# ------------------------------------------------------------

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

gc.collect()

e5_model = SentenceTransformer(E5_MODEL_NAME, device=DEVICE)

# ------------------------------------------------------------
# Compute lexical + semantic metrics
# ------------------------------------------------------------

summary_rows = []

for exp in experiments:
    label = exp["label"]

    print("\n" + "=" * 90)
    print("Processing:", label)
    print("=" * 90)

    if not exp["found"]:
        summary_rows.append({
            "experiment": label,
            "experiment_name": "",
            "model_type": exp["model_type"],
            "found": False,
            "num_examples": None,
            "unique_source_ids": None,
            "BLEU": None,
            "spBLEU": None,
            "spBLEU_tokenizer": None,
            "chrF": None,
            "chrF++": None,
            "E5_large_cosine_mean": None,
            "E5_large_cosine_median": None,
            "E5_large_cosine_std": None,
            "E5_large_cosine_p05": None,
            "E5_large_cosine_p95": None,
            "best_step": None,
            "best_eval_loss": None,
            "prediction_file": exp["prediction_file"],
            "metrics_json": exp["metrics_path"],
            "semantic_json": None,
            "semantic_predictions": None,
            "error": exp["error"],
        })
        print("Skipped:", exp["error"])
        continue

    pred_file = Path(exp["prediction_file"])
    df = load_prediction_df(pred_file)

    print("Prediction file:", pred_file)
    print("Rows after deduplication:", len(df))
    print("Unique source_ids:", df["source_id"].nunique())

    mt_scores = compute_mt_metrics_from_df(df)

    print("BLEU:", mt_scores["BLEU"])
    print("spBLEU:", mt_scores["spBLEU"])
    print("spBLEU tokenizer:", mt_scores["spBLEU_tokenizer"])
    if mt_scores["spBLEU_error"]:
        print("spBLEU warning:", mt_scores["spBLEU_error"])
    print("chrF:", mt_scores["chrF"])
    print("chrF++:", mt_scores["chrF++"])

    sims = compute_e5_pairwise_similarity(df, e5_model)
    df["E5_large_cosine"] = sims

    semantic_scores = summarize_similarity(sims)

    metrics_data = exp.get("metrics_data") or {}

    experiment_name = metrics_data.get(
        "experiment",
        safe_first(df, "experiment_name", infer_experiment_name_from_prediction_path(pred_file))
    )

    display_name = label

    semantic_pred_path = SEMANTIC_OUT_DIR / f"semantic_predictions_e5_large_{pred_file.stem}.csv"
    semantic_json_path = SEMANTIC_OUT_DIR / f"semantic_metrics_e5_large_{pred_file.stem}.json"

    df.to_csv(semantic_pred_path, index=False, encoding="utf-8-sig")

    best_step = metrics_data.get("best_step", safe_first(df, "best_step", None))
    best_eval_loss = metrics_data.get("best_eval_loss", safe_first(df, "best_eval_loss", None))
    checkpoint = metrics_data.get("checkpoint", safe_first(df, "model_checkpoint", ""))

    full_report = {
        "experiment": experiment_name,
        "display_name": display_name,
        "model_type": exp["model_type"],
        "type": exp["model_type"],

        "checkpoint": checkpoint,
        "best_step": best_step,
        "best_eval_loss": best_eval_loss,

        "semantic_model": E5_MODEL_NAME,
        "semantic_metric": "pairwise cosine similarity between reference_arabic and prediction",

        "num_examples": int(len(df)),
        "unique_source_ids": int(df["source_id"].nunique()),

        **mt_scores,
        **semantic_scores,

        "prediction_file": str(pred_file),
        "semantic_prediction_file": str(semantic_pred_path),
        "semantic_metrics_file": str(semantic_json_path),
        "source_metrics_path": exp["metrics_path"],

        "model_name": safe_first(df, "model_name", ""),
        "baseline_label": safe_first(df, "baseline_label", ""),
        "template_mode": safe_first(df, "template_mode", ""),

        "per_config": compute_group_metrics(df, "config"),
        "per_dialect": compute_group_metrics(df, "dialect"),
        "per_domain": compute_group_metrics(df, "domain"),
    }

    write_json(semantic_json_path, full_report)

    # --------------------------------------------------------
    # Save standard full_eval_metrics JSON consistently
    # --------------------------------------------------------

    if exp["model_type"] == "baseline_no_finetune":
        standard_metrics_json_path = infer_baseline_metrics_path(experiment_name)
    else:
        if exp["metrics_path"]:
            standard_metrics_json_path = Path(exp["metrics_path"])
        else:
            standard_metrics_json_path = derive_metrics_path_from_prediction_path(pred_file)

    write_json(standard_metrics_json_path, full_report)

    summary_rows.append({
        "experiment": display_name,
        "experiment_name": experiment_name,
        "model_type": exp["model_type"],
        "found": True,

        "num_examples": full_report["num_examples"],
        "unique_source_ids": full_report["unique_source_ids"],

        "BLEU": mt_scores["BLEU"],
        "spBLEU": mt_scores["spBLEU"],
        "spBLEU_tokenizer": mt_scores["spBLEU_tokenizer"],
        "chrF": mt_scores["chrF"],
        "chrF++": mt_scores["chrF++"],

        "E5_large_cosine_mean": semantic_scores["E5_large_cosine_mean"],
        "E5_large_cosine_median": semantic_scores["E5_large_cosine_median"],
        "E5_large_cosine_std": semantic_scores["E5_large_cosine_std"],
        "E5_large_cosine_p05": semantic_scores["E5_large_cosine_p05"],
        "E5_large_cosine_p95": semantic_scores["E5_large_cosine_p95"],

        "best_step": best_step,
        "best_eval_loss": best_eval_loss,
        "checkpoint": checkpoint,

        "prediction_file": str(pred_file),
        "metrics_json": str(standard_metrics_json_path),
        "semantic_json": str(semantic_json_path),
        "semantic_predictions": str(semantic_pred_path),
        "error": "",
    })

    print("Mean E5-large cosine:", semantic_scores["E5_large_cosine_mean"])
    print("Saved standard metrics JSON:", standard_metrics_json_path)
    print("Saved semantic predictions:", semantic_pred_path)
    print("Saved semantic report:", semantic_json_path)

# ------------------------------------------------------------
# Final comparison table
# ------------------------------------------------------------

semantic_comparison_df = pd.DataFrame(summary_rows)

numeric_cols = [
    "BLEU",
    "spBLEU",
    "chrF",
    "chrF++",
    "E5_large_cosine_mean",
    "E5_large_cosine_median",
    "E5_large_cosine_std",
    "E5_large_cosine_p05",
    "E5_large_cosine_p95",
    "best_eval_loss",
]

for col in numeric_cols:
    if col in semantic_comparison_df.columns:
        semantic_comparison_df[col] = pd.to_numeric(
            semantic_comparison_df[col],
            errors="coerce",
        )

semantic_comparison_df = semantic_comparison_df.sort_values(
    by=["found", "chrF++", "spBLEU", "BLEU"],
    ascending=[False, False, False, False],
    na_position="last",
).reset_index(drop=True)

comparison_csv_path = SEMANTIC_OUT_DIR / "comparison_semantic_similarity_e5_large_qwen_gemma_baselines.csv"
comparison_json_path = SEMANTIC_OUT_DIR / "comparison_semantic_similarity_e5_large_qwen_gemma_baselines.json"

semantic_comparison_df.to_csv(
    comparison_csv_path,
    index=False,
    encoding="utf-8-sig",
)

comparison_records = semantic_comparison_df.to_dict("records")

write_json(
    comparison_json_path,
    {
        "comparison_name": "qwen_gemma_baselines_and_finetuned_e5_large",
        "semantic_model": E5_MODEL_NAME,
        "spBLEU_tokenizer": SPBLEU_TOKENIZER,
        "num_systems": int(len(semantic_comparison_df)),
        "systems": comparison_records,
    },
)

print("\n" + "=" * 90)
print("Final Lexical + Semantic Comparison using E5-large")
print("=" * 90)

display_cols = [
    "experiment",
    "model_type",
    "found",
    "num_examples",
    "BLEU",
    "spBLEU",
    "chrF",
    "chrF++",
    "E5_large_cosine_mean",
    "E5_large_cosine_median",
    "E5_large_cosine_std",
    "E5_large_cosine_p05",
    "E5_large_cosine_p95",
    "best_step",
    "best_eval_loss",
    "error",
]

display(semantic_comparison_df[display_cols])

print("\nFiles saved:")
display(
    semantic_comparison_df[
        [
            "experiment",
            "prediction_file",
            "metrics_json",
            "semantic_json",
            "semantic_predictions",
        ]
    ]
)

print("\nSaved final comparison CSV:")
print(comparison_csv_path)

print("\nSaved final comparison JSON:")
print(comparison_json_path)

Missing packages: []
PROJECT_DIR: /content/drive/MyDrive/alexandria_qwen35_sft
FINETUNED_PRED_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/predictions
BASELINE_PRED_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/baselines/predictions
SEMANTIC_OUT_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/semantic_similarity_e5_large
Using device: cuda
E5 model: intfloat/multilingual-e5-large
Batch size: 8
spBLEU tokenizer: flores200

Experiments to evaluate:
- Qwen3.5-2B LoRA all-r16 best_step600 | found: True | pred: /content/drive/MyDrive/alexandria_qwen35_sft/predictions/full_eval_predictions_qwen35_2b_alexandria_eg_only_context3_all_group_r16_v2_10epochs_best_step600.csv
- Gemma-4-E2B-it FNN-r8 MLP best_step2500 | found: True | pred: /content/drive/MyDrive/alexandria_qwen35_sft/predictions/full_eval_predictions_gemma4_e2b_it_alexandria_eg_only_context3_fnn_group_r8_prompt_nat_eg_mlp_r8_lr1e5_10epochs_v1_best_step2500.csv
- Qwen3-4B LoRA complete-2shot all-r16 best_step700 | found: T

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

Mean E5-large cosine: 0.9298944278671831
Saved standard metrics JSON: /content/drive/MyDrive/alexandria_qwen35_sft/baselines/predictions/full_eval_metrics_baseline_no_finetune_qwen3_4b_instruct_complete2shot_chat_v1_alexandria_eg_only_context3.json
Saved semantic predictions: /content/drive/MyDrive/alexandria_qwen35_sft/semantic_similarity_e5_large/semantic_predictions_e5_large_full_eval_predictions_baseline_no_finetune_qwen3_4b_instruct_complete2shot_chat_v1_alexandria_eg_only_context3.csv
Saved semantic report: /content/drive/MyDrive/alexandria_qwen35_sft/semantic_similarity_e5_large/semantic_metrics_e5_large_full_eval_predictions_baseline_no_finetune_qwen3_4b_instruct_complete2shot_chat_v1_alexandria_eg_only_context3.json

Final Lexical + Semantic Comparison using E5-large


,experiment,model_type,found,num_examples,BLEU,spBLEU,chrF,chrF++,E5_large_cosine_mean,E5_large_cosine_median,E5_large_cosine_std,E5_large_cosine_p05,E5_large_cosine_p95,best_step,best_eval_loss,error
0,baseline_no_finetune_gemma4_e2b_it_base_alexan...,baseline_no_finetune,True,1118,14.545134,26.848712,44.957983,41.905887,0.952464,0.954937,0.023460,0.910387,0.986728,NaN,NaN,
1,Gemma-4-E2B-it FNN-r8 MLP best_step2500,finetuned_lora,True,1118,13.369876,25.228667,43.155348,40.144052,0.949918,0.951944,0.023773,0.908785,0.984998,2500.0,2.918498,
2,Qwen3-4B LoRA complete-2shot all-r16 best_step700,finetuned_lora,True,1118,10.108711,19.695814,38.163383,35.439305,0.944603,0.946915,0.024653,0.901456,0.981440,700.0,1.605848,
3,Qwen3-4B LoRA no-shot all-r16 best_step700,finetuned_lora,True,1118,9.825838,19.517034,37.894180,35.052684,0.944039,0.945949,0.025108,0.900874,0.980855,700.0,1.605091,
4,Qwen3.5-2B LoRA all-r16 best_step600,finetuned_lora,True,1118,8.561227,18.592952,37.912548,34.572649,0.943186,0.944604,0.024783,0.897434,0.980958,600.0,2.015077,
5,baseline_no_finetune_qwen3_4b_instruct_complet...,baseline_no_finetune,True,1118,3.193701,9.846157,28.436726,25.111000,0.929894,0.932234,0.027395,0.882666,0.970839,NaN,NaN,
6,baseline_no_finetune_qwen3_4b_base_alexandria_...,baseline_no_finetune,True,1118,2.588964,8.101324,26.765252,23.478766,0.924230,0.927411,0.031159,0.864695,0.968886,NaN,NaN,
7,baseline_no_finetune_qwen35_2b_base_alexandria...,baseline_no_finetune,True,1118,2.366042,7.814773,26.516454,23.239397,0.923324,0.927913,0.032006,0.866577,0.968738,NaN,NaN,
8,baseline_no_finetune_qwen3_4b_base_complete2sh...,baseline_no_finetune,True,1118,2.328091,7.355272,25.910197,22.800782,0.920056,0.923959,0.033236,0.856552,0.966432,NaN,NaN,



Files saved:


,experiment,prediction_file,metrics_json,semantic_json,semantic_predictions
0,baseline_no_finetune_gemma4_e2b_it_base_alexan...,/content/drive/MyDrive/alexandria_qwen35_sft/b...,/content/drive/MyDrive/alexandria_qwen35_sft/b...,/content/drive/MyDrive/alexandria_qwen35_sft/s...,/content/drive/MyDrive/alexandria_qwen35_sft/s...
1,Gemma-4-E2B-it FNN-r8 MLP best_step2500,/content/drive/MyDrive/alexandria_qwen35_sft/p...,/content/drive/MyDrive/alexandria_qwen35_sft/p...,/content/drive/MyDrive/alexandria_qwen35_sft/s...,/content/drive/MyDrive/alexandria_qwen35_sft/s...
2,Qwen3-4B LoRA complete-2shot all-r16 best_step700,/content/drive/MyDrive/alexandria_qwen35_sft/p...,/content/drive/MyDrive/alexandria_qwen35_sft/p...,/content/drive/MyDrive/alexandria_qwen35_sft/s...,/content/drive/MyDrive/alexandria_qwen35_sft/s...
3,Qwen3-4B LoRA no-shot all-r16 best_step700,/content/drive/MyDrive/alexandria_qwen35_sft/p...,/content/drive/MyDrive/alexandria_qwen35_sft/p...,/content/drive/MyDrive/alexandria_qwen35_sft/s...,/content/drive/MyDrive/alexandria_qwen35_sft/s...
4,Qwen3.5-2B LoRA all-r16 best_step600,/content/drive/MyDrive/alexandria_qwen35_sft/p...,/content/drive/MyDrive/alexandria_qwen35_sft/p...,/content/drive/MyDrive/alexandria_qwen35_sft/s...,/content/drive/MyDrive/alexandria_qwen35_sft/s...
5,baseline_no_finetune_qwen3_4b_instruct_complet...,/content/drive/MyDrive/alexandria_qwen35_sft/b...,/content/drive/MyDrive/alexandria_qwen35_sft/b...,/content/drive/MyDrive/alexandria_qwen35_sft/s...,/content/drive/MyDrive/alexandria_qwen35_sft/s...
6,baseline_no_finetune_qwen3_4b_base_alexandria_...,/content/drive/MyDrive/alexandria_qwen35_sft/b...,/content/drive/MyDrive/alexandria_qwen35_sft/b...,/content/drive/MyDrive/alexandria_qwen35_sft/s...,/content/drive/MyDrive/alexandria_qwen35_sft/s...
7,baseline_no_finetune_qwen35_2b_base_alexandria...,/content/drive/MyDrive/alexandria_qwen35_sft/b...,/content/drive/MyDrive/alexandria_qwen35_sft/b...,/content/drive/MyDrive/alexandria_qwen35_sft/s...,/content/drive/MyDrive/alexandria_qwen35_sft/s...
8,baseline_no_finetune_qwen3_4b_base_complete2sh...,/content/drive/MyDrive/alexandria_qwen35_sft/b...,/content/drive/MyDrive/alexandria_qwen35_sft/b...,/content/drive/MyDrive/alexandria_qwen35_sft/s...,/content/drive/MyDrive/alexandria_qwen35_sft/s...



Saved final comparison CSV:
/content/drive/MyDrive/alexandria_qwen35_sft/semantic_similarity_e5_large/comparison_semantic_similarity_e5_large_qwen_gemma_baselines.csv

Saved final comparison JSON:
/content/drive/MyDrive/alexandria_qwen35_sft/semantic_similarity_e5_large/comparison_semantic_similarity_e5_large_qwen_gemma_baselines.json
